In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from tqdm import tqdm
import re
import pandas as pd
import os
import seaborn as sns
from scipy.signal import medfilt
from concurrent.futures import ThreadPoolExecutor, as_completed
import shutil

In [2]:
NTU_RAW_DATA_DIR = "./NTU_raw_npy/"
SPINE_BASE = 0
SPINE_MID = 1
TORSO_JOINTS = [0, 1, 2, 3, 20]
LIMB_JOINTS = [7, 11, 15, 19]
NTU_INTERACTIVE_ACTION_IDS = set(range(50, 61)) | set(range(106, 121))
NTU_JOINT_NAMES = {
    0: "SpineBase", 1: "SpineMid", 2: "Neck", 3: "Head",
    4: "ShoulderL", 5: "ElbowL", 6: "WristL", 7: "HandL",
    8: "ShoulderR", 9: "ElbowR", 10: "WristR", 11: "HandR",
    12: "HipL", 13: "KneeL", 14: "AnkleL", 15: "FootL",
    16: "HipR", 17: "KneeR", 18: "AnkleR", 19: "FootR",
    20: "SpineShoulder", 21: "HandTipL", 22: "ThumbL", 23: "HandTipR", 24: "ThumbR"
}
NTU_ACTIONS = {
    "001": "drink water", "002": "eat meal/snack", "003": "brushing teeth", "004": "brushing hair", 
    "005": "drop", "006": "pickup", "007": "throw", "008": "sitting down", "009": "standing up (from sitting position)", 
    "010": "clapping", "011": "reading", "012": "writing", "013": "tear up paper", "014": "wear jacket", 
    "015": "take off jacket", "016": "wear a shoe", "017": "take off a shoe", "018": "wear on glasses", 
    "019": "take off glasses", "020": "put on a hat/cap", "021": "take off a hat/cap", "022": "cheer up", 
    "023": "hand waving", "024": "kicking something", "025": "reach into pocket", "026": "hopping (one foot jumping)", 
    "027": "jump up", "028": "make a phone call/answer phone", "029": "playing with phone/tablet", 
    "030": "typing on a keyboard", "031": "pointing to something with finger", "032": "taking a selfie", 
    "033": "check time (from watch)", "034": "rub two hands together", "035": "nod head/bow", 
    "036": "shake head", "037": "wipe face", "038": "salute", "039": "put the palms together", 
    "040": "cross hands in front (say stop)", "041": "sneeze/cough", "042": "staggering", "043": "falling", 
    "044": "touch head (headache)", "045": "touch chest (stomachache/heart pain)", "046": "touch back (backache)", 
    "047": "touch neck (neckache)", "048": "nausea or vomiting condition", "049": "use a fan (with hand or paper)/feeling warm", 
    "050": "punching/slapping other person", "051": "kicking other person", "052": "pushing other person", 
    "053": "pat on back of other person", "054": "point finger at the other person", "055": "hugging other person", 
    "056": "giving something to other person", "057": "touch other person's pocket", "058": "handshaking", 
    "059": "walking towards each other", "060": "walking apart from each other", "061": "put on headphone", 
    "062": "take off headphone", "063": "shoot at the basket", "064": "bounce ball", "065": "tennis bat swing", 
    "066": "juggling table tennis balls", "067": "hush (quite)", "068": "flick hair", "069": "thumb up", 
    "070": "thumb down", "071": "make ok sign", "072": "make victory sign", "073": "staple book", 
    "074": "counting money", "075": "cutting nails", "076": "cutting paper (using scissors)", "077": "snapping fingers", 
    "078": "open bottle", "079": "sniff (smell)", "080": "squat down", "081": "toss a coin", "082": "fold paper", 
    "083": "ball up paper", "084": "play magic cube", "085": "apply cream on face", "086": "apply cream on hand back", 
    "087": "put on bag", "088": "take off bag", "089": "put something into a bag", "090": "take something out of a bag", 
    "091": "open a box", "092": "move heavy objects", "093": "shake fist", "094": "throw up cap/hat", 
    "095": "hands up (both hands)", "096": "cross arms", "097": "arm circles", "098": "arm swings", 
    "099": "running on the spot", "100": "butt kicks (kick backward)", "101": "cross toe touch", 
    "102": "side kick", "103": "yawn", "104": "stretch oneself", "105": "blow nose", "106": "hit other person with something", 
    "107": "wield knife towards other person", "108": "knock over other person (hit with body)", 
    "109": "grab other person’s stuff", "110": "shoot at other person with a gun", "111": "step on foot", 
    "112": "high-five", "113": "cheers and drink", "114": "carry something with other person", 
    "115": "take a photo of other person", "116": "follow other person", "117": "whisper in other person’s ear", 
    "118": "exchange things with other person", "119": "support somebody with hand", 
    "120": "finger-guessing game (playing rock-paper-scissors)"
}

In [3]:
def ntu_filename_to_meanings(file_name:str):
    # verify file ending and type
    if file_name[-13:] != ".skeleton.npy":
        print(f"Provided filename has incorrect endings and/or typing -> {file_name}")
        return None
    else:
        arr = file_name[:4],file_name[4:8],file_name[8:12],file_name[12:16],file_name[16:20]  
        return [n[1:] for n in arr]

def animate_ntu_multi_bodies(file_path):
    """
    Generate a 3D animation of NTU RGB+D skeleton data from a .npy file.

    This function parses the action label from the filename, extracts up to
    four skeleton bodies from the data dictionary, and renders an interactive
    HTML5/JavaScript animation with bone connections and distinct colors.

    Args:
        file_path (str): Path to the .skeleton.npy file.

    Returns:
        IPython.core.display.HTML: An interactive animation widget for Notebooks.
        str: An error message if no active skeletons are found.
    """
    # 1. Setup Action Labeling
    # Extract the 'Axxx' code to look up the human-readable action name
    match = re.search(r'A(\d{3})', file_path)
    action_code = match.group(1) if match else "000"
    action_name = NTU_ACTIONS.get(action_code, "Unknown Action")

    # 2. Extract Dictionaries and Filter Active Bodies
    # NTU data is often saved as a dictionary containing 'skel_bodyX' keys
    data = np.load(file_path, allow_pickle=True).item()
    
    # Check for skel_body0 through skel_body3 (rare in most setups)
    body_keys = [k for k in data.keys() if 'skel_body' in k]
    active_bodies = []
    
    for k in sorted(body_keys):
        skeleton = data[k]
        # Only keep the skeleton if it contains tracked data (non-zero)
        if np.any(skeleton != 0):
            active_bodies.append(skeleton)

    if not active_bodies:
        return "No skeleton data found in file."

    num_frames = active_bodies[0].shape[0]

    # 3. Setup 3D Plot
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Standard Kinect V2 Bone Map (1-indexed converted to 0-indexed)
    bone_map = [
        (1, 2), (2, 21), (21, 3), (3, 4), (21, 5), (5, 6), (6, 7), (7, 8),
        (21, 9), (9, 10), (10, 11), (11, 12), (1, 13), (13, 14), (14, 15),
        (15, 16), (1, 17), (17, 18), (18, 19), (19, 20), (22, 23), (22, 8),
        (24, 25), (24, 12)
    ]
    bones = [(i - 1, j - 1) for i, j in bone_map]

    # 4. Initialize Plot Elements
    # Define distinct color palettes for multi-person interactions
    body_colors = ['red', 'green', 'purple', 'orange']
    bone_colors = ['blue', 'darkgreen', 'indigo', 'darkorange']
    
    scatters = []
    bone_lines = []

    for i in range(len(active_bodies)):
        scatters.append(ax.scatter([], [], [], 
                                   c=body_colors[i % 4], 
                                   s=15, 
                                   label=f'Body {i}'))
        bone_lines.append([ax.plot([], [], [], 
                                   c=bone_colors[i % 4], 
                                   alpha=0.6)[0] for _ in bones])

    # Plot Aesthetics and Labels
    # Kinect Space: X=Lateral, Y=Vertical, Z=Depth. 
    # We map Kinect Y to Plot Z to keep the skeleton standing upright.
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(0, 5)   # Depth
    ax.set_zlim(-1, 1)  # Height
    ax.set_xlabel("X (Lateral)")
    ax.set_ylabel("Z (Depth)")
    ax.set_zlabel("Y (Vertical)")
    ax.legend(loc='upper right')
    
    plt.title(f"NTU {action_code}: {action_name.upper()}", 
              fontsize=14, pad=20)
    ax.view_init(elev=15, azim=110)

    def update(frame):
        """Update function for FuncAnimation loop."""
        artists = []
        for b_idx, skel in enumerate(active_bodies):
            joints = skel[frame]
            
            # Update joint positions (X, Z, Y)
            scatters[b_idx]._offsets3d = (joints[:, 0], joints[:, 2], joints[:, 1])
            
            # Update bone line positions
            for i, (b1, b2) in enumerate(bones):
                bone_lines[b_idx][i].set_data([joints[b1, 0], joints[b2, 0]], 
                                              [joints[b1, 2], joints[b2, 2]])
                bone_lines[b_idx][i].set_3d_properties([joints[b1, 1], joints[b2, 1]])
            
            artists += [scatters[b_idx]] + bone_lines[b_idx]
        return artists

    # 5. Create and Return Animation
    # tqdm is used to track the rendering progress of each frame
    ani = FuncAnimation(
        fig, 
        update, 
        frames=tqdm(range(num_frames), desc=f"Processing {action_code} - {action_name}"), 
        interval=33, 
        blit=True
    )
    
    plt.close()  # Prevents display of a redundant static plot
    return HTML(ani.to_jshtml())



In [4]:
# Animate a .npy file
# animate_ntu_multi_bodies(f"{NTU_RAW_DATA_DIR}{f_name}")

In [5]:
"""
DATA STRUCT SAMPLE
data = Dict{
file_name:str, file name
nbodys:List[int], number of bodies for each frame
njoints:Constant int, constant value, number of joints
skel_bodyn: (nframe, njoint, 3) where n is person id
rgb_bodyn: 2D skeleton projection in RGBs
depth_bodyn: 2D skeleton projection in Depth
}
.npy FILE NAME DEFINITIONS
.npy data formats: S001C001P001R001A001.skeleton.npy where...
S = Setup ID, C = Camera ID, P = Performer ID, R = Replication Number (1 or 2), A = Action ID

Auditting data

Actions
A0~49, A61~A105: Single body actions
A50~60, A106~A120: 2 body interactive actions
 - Get total number of Action IDs in dataset
 - Plot number of "single body actions" vs number of "2 body interactive actions"
 - Plot number of data labelled with "single body actions" which has at least 1 frame with 2 or more bodies present
 - Plot number of data labelled with "2 body interactive actions" which has at least 1 frame with 2 or more bodies present

Setup
 - Get total number of different setup IDs in dataset
 - Plot Setup IDs against number of data (single file of .npy) which uses given Setup ID

Camera
 - Get total number of different Camera IDs in dataset
 - Plot Camera IDs against number of data (single file of .npy) which uses given Camera ID

Performer
 - Get total number of different Performer IDs in dataset
 - Plot Performer IDs against number of data (single file of .npy) which uses given Performer ID

Frame Size
 - Plot to show the distribution of frame sizes for each data

Validity of xxx_bodyn keys and the n_joints key
 - Get number of data (single file of .npy) files where skel_bodyn, rgb_bodyn and depth_bodyn's keys exist, frames match out of the total number 
 of data and have matching frame counts
 - Get number of data with missing joints (invalid joint values)
 - Get number of data with impossibly long bone lengths (distance between 2 connected joints) by comparing to known spine value
 - Get number of data with discontinued (unusualy) temporal continuity by ensuring movement of any joint is not too rapid
"""

def audit_ntu_structure(root_dir):
    results = []
    TORSO_THRESHOLD = 0.3  # ~9.0 m/s
    LIMB_THRESHOLD = 0.5   # ~15 m/s
    # Check SpineBase (Index 0), SpineMid (1), Neck (2), Head (3)
    TORSO_IDXS = [0, 1, 2, 3, 20] 
    # Check Hands (7, 11) and Feet (15, 19)
    LIMB_IDXS = [7, 11, 15, 19]
    
    file_list = [f for f in os.listdir(root_dir) if f.endswith('.skeleton.npy')]
    
    for fname in tqdm(file_list, desc="Deep Audit"):
        # Parse Filename Metadata
        # S001C001P001R001A001
        match = re.search(r'S(\d{3})C(\d{3})P(\d{3})R(\d{3})A(\d{3})', fname)
        if not match: continue
        
        s_id, c_id, p_id, r_id, a_id = map(int, match.groups())
        is_interaction_label = a_id in NTU_INTERACTIVE_ACTION_IDS
        
        fpath = os.path.join(root_dir, fname)
        try:
            data = np.load(fpath, allow_pickle=True).item()
            
            # --- Body Logic Checks ---
            # nbodys is a list/array of body counts per frame
            nbodys = np.array(data.get('nbodys', []))
            max_bodies_detected = np.max(nbodys) if len(nbodys) > 0 else 0
            has_multiple_bodies = max_bodies_detected >= 2
            
            # --- Key Validity & Frame Consistency ---
            # Check existence of keys for body0 (primary)
            keys_exist = all(k in data for k in ['skel_body0', 'rgb_body0', 'depth_body0'])
            frames_match = False
            if keys_exist:
                f_skel = data['skel_body0'].shape[0]
                f_rgb = data['rgb_body0'].shape[0]
                f_depth = data['depth_body0'].shape[0]
                frames_match = (f_skel == f_rgb == f_depth == len(nbodys))
            
            # --- Physical/Temporal Sanity (Primary Body) ---
            skel = data.get('skel_body0', None)
            invalid_joints = False
            broken_bones = False
            rapid_torso_movement = False
            rapid_limbs_movement = False
            
            if skel is not None and skel.size > 0:
                # 1. Invalid Joints (Checking for 0,0,0 or NaNs)
                if np.any(np.all(skel == 0, axis=-1)) or np.any(np.isnan(skel)):
                    invalid_joints = True
                
                # 2. Impossible Bone Lengths (SpineMid [1] to SpineBase [0])
                # Reference: average spine is ~0.3-0.5 meters. 
                # If distance > 1.0m or < 0.05m, the track is broken.
                spine_vec = skel[:, 1, :] - skel[:, 0, :]
                spine_len = np.linalg.norm(spine_vec, axis=1)
                if np.any(spine_len > 1.0) or np.any(spine_len < 0.05):
                    broken_bones = True
                
                # 3. Temporal Continuity (Max velocity)
                if skel.shape[0] > 1:
                    velocity = np.linalg.norm(np.diff(skel, axis=0), axis=-1)
                    too_fast_torso = np.any(velocity[:, TORSO_IDXS] > TORSO_THRESHOLD)
                    too_fast_limbs = np.any(velocity[:, LIMB_IDXS] > LIMB_THRESHOLD)
                    if too_fast_torso: rapid_torso_movement = True
                    if too_fast_limbs: rapid_limbs_movement = True

            results.append({
                'fname': fname,
                'S': s_id, 'C': c_id, 'P': p_id, 'A': a_id,
                'is_interaction_label': is_interaction_label,
                'has_multiple_bodies': has_multiple_bodies,
                'n_frames': len(nbodys),
                'keys_valid': keys_exist and frames_match,
                'invalid_joints': invalid_joints,
                'broken_bones': broken_bones,
                'rapid_torso_movement': rapid_torso_movement,
                'rapid_limbs_movement': rapid_limbs_movement
            })
            
        except Exception:
            continue

    return pd.DataFrame(results)

# df_audit = audit_ntu_structure(NTU_RAW_DATA_DIR)
# df_audit.to_csv("NTU_visualization.csv", index=False)

# df_audit = pd.read_csv("NTU_visualization.csv")

def plot_audit_results(df):
    sns.set_theme(style="whitegrid")
    fig = plt.figure(figsize=(20, 24))
    
    # 1. Single vs Interaction Label Count
    ax1 = plt.subplot(3, 3, 1)
    sns.countplot(data=df, x='is_interaction_label', hue='is_interaction_label', palette='Set2', ax=ax1, legend=False)
    ax1.set_title("Single Body Actions (False) vs Interaction Actions (True)")
    
    # 2. Logic Check: Labels vs Detected Bodies
    ax2 = plt.subplot(3, 3, 2)
    # Filter for cases where 2+ bodies were actually found
    sns.countplot(data=df, x='is_interaction_label', hue='has_multiple_bodies', ax=ax2)
    ax2.set_title("Labeled Action vs. At least 1 frame with 2+ Bodies")

    # Distribution of Action IDs
    ax3 = plt.subplot(3, 3, 3)
    sns.countplot(data=df, x="A", color='steelblue', ax=ax3)
    ax3.set_title(f"Data count per setup ID (Total: {df["A"].nunique()})")
    
    # 3. Setup IDs
    ax4 = plt.subplot(3, 3, 4)
    sns.countplot(data=df, x='S', color='steelblue', ax=ax4)
    ax4.set_title(f"Data count per Setup ID (Total: {df['S'].nunique()})")
    
    # 4. Camera IDs
    ax5 = plt.subplot(3, 3, 5)
    sns.countplot(data=df, x='C', color='seagreen', ax=ax5)
    ax5.set_title(f"Data count per Camera ID (Total: {df['C'].nunique()})")
    
    # 5. Performer IDs
    ax6 = plt.subplot(3, 3, 6)
    sns.countplot(data=df, x='P', color='indianred', ax=ax6)
    ax6.set_title(f"Data count per Performer ID (Total: {df['P'].nunique()})")
    plt.xticks(rotation=90)

    # 6. Frame Size Distribution
    ax7 = plt.subplot(3, 3, 7)
    sns.histplot(df['n_frames'], bins=50, kde=True, ax=ax7)
    ax7.set_title("Distribution of Frame Sizes (Sequence Length)")

    # 7. Validity Summary (Key/Frame consistency)
    ax8 = plt.subplot(3, 3, 8)
    valid_counts = df['keys_valid'].value_counts()
    ax8.pie(valid_counts, labels=valid_counts.index, autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
    ax8.set_title("Keys Exist & Frames Match")

    # 8. Physical Anomaly Counts
    ax9 = plt.subplot(3, 3, 9)
    anomalies = {
        'Invalid Joints': df['invalid_joints'].sum(),
        'Broken Bones': df['broken_bones'].sum(),
        'Rapid Torso Motion': df['rapid_torso_movement'].sum(),
        'Rapid Limbs Motion': df['rapid_limbs_movement'].sum()
    }
    plt.bar(anomalies.keys(), anomalies.values(), color='orange')
    ax9.set_title("Physical Data Violations")

    plt.tight_layout()
    plt.show()

# print(f"Number of data (rows): {df_audit.shape[0]}")
# plot_audit_results(df_audit)

In [6]:
"""
Data Preprocessing
Points with * marks means it is not necessary atm.

Data Cleaning
 - For single action labelled data which has at least 1 frame of multiple body presence
    - Ensure the labelled action is not on the sub-body (body which appears for shorter time, frame-wise not dominant in data)
    - Remove the _noise_ body from it's data using temporal coordinate tracking, joint variance changes
 - For interactive action labelled data which does not have majority of its frames with 2 bodies
    - Remove data where less than 2 bodies are present for 50% of the time
 - Most data have frame length of 70-90
    - Remove outliers out of 30-200
 - Torso too fast
    - Remove data

Interpolating
 - Invalid joints (joint value is (0,0,0) or NaN)
    - Interpolate with temporal vector, else remove frame unless relevant
 - Spine-middle to spine-bottom may be too short or too long
    - Interpolate with temporal vector, else remove frame unless relevant
 - Limbs too fast
    - Interpolate with temporal vector, else remove frame unless relevant
    
Skews
 - Way more single body labelled actions than interactive actions
    - Down sample single body actions. When downsampling, for given action ID, try choose data with highest joint variance (most movement)
 - * Disproportionate Setup ID's and its data count
    - Randomly sample from each setup ID
 - * Disproportionate Performer ID's and its data count (Only 1 performer ID is displayed but there may be multiple bodies in frame)
    - Randomly sample from each performer ID

Prep
 - Normalize target body/bodies to be in center of env (normalize joints)
"""

# --- Configuration ---
PROC_DIR = "./NTU_preprocessed/"
REGISTRY_PATH = "ntu_preprocess_master_registry.csv"
CORE_USAGE_ALLOWED = os.cpu_count()*2 # *2 for threads, //2 for parallelization
os.makedirs(PROC_DIR, exist_ok=True)

def reset_preprocessing_env():
    # 1. Remove the Registry CSV
    if os.path.exists(REGISTRY_PATH):
        try:
            os.remove(REGISTRY_PATH)
            print(f"✅ Successfully removed registry: {REGISTRY_PATH}")
        except Exception as e:
            print(f"❌ Error removing {REGISTRY_PATH}: {e}")
    else:
        print(f"ℹ️ Registry file not found at {REGISTRY_PATH}, skipping.")

    # 2. Clear the Processed Directory
    if os.path.exists(PROC_DIR):
        try:
            # We use shutil.rmtree to remove the folder and all contents
            # then recreate it empty.
            shutil.rmtree(PROC_DIR)
            os.makedirs(PROC_DIR)
            print(f"✅ Successfully cleared and recreated directory: {PROC_DIR}")
        except Exception as e:
            print(f"❌ Error clearing {PROC_DIR}: {e}")
    else:
        os.makedirs(PROC_DIR)
        print(f"ℹ️ Processed directory created at {PROC_DIR}")

def process_single_file_metadata(fname):
    """
    Worker function: Handles the parsing and data extraction for one file.
    """
    match = re.search(r'S(\d{3})C(\d{3})P(\d{3})R(\d{3})A(\d{3})', fname)
    if not match:
        return None

    s, c, p, r, a = map(int, match.groups())
    fpath = os.path.join(NTU_RAW_DATA_DIR, fname)
    
    try:
        # allow_pickle=True is required for NTU .npy files
        data = np.load(fpath, allow_pickle=True).item()
        nbodys = data.get('nbodys', [])
        n_frames = len(nbodys)
        max_bodies = np.max(nbodys) if n_frames > 0 else 0
        
        too_fast_torso = False
        skel = data.get('skel_body0')
        
        if skel is not None and skel.shape[0] > 1:
            # Indices: SpineBase(0), SpineMid(1), Neck(2), Head(3), SpineShoulder(20)
            torso_v = np.linalg.norm(np.diff(skel[:, [0,1,2,3,20], :], axis=0), axis=-1)
            if np.any(torso_v > 0.15): 
                too_fast_torso = True

        return {
            'file_name': fname, 'A': a, 'P': p, 'S': s,
            'n_frames': n_frames, 'max_bodies': max_bodies,
            'too_fast_torso': too_fast_torso,
            'is_interaction': a in NTU_INTERACTIVE_ACTION_IDS
        }
    except Exception as e:
        return f"Error reading {fname}: {e}"

def build_master_registry_parallel(max_workers):
    if os.path.exists(REGISTRY_PATH):
        print("Loading existing registry...")
        return pd.read_csv(REGISTRY_PATH)
    print(f"\n>>> Building master registry - Pipeline ({max_workers} threads)")

    file_list = [f for f in os.listdir(NTU_RAW_DATA_DIR) if f.endswith('.skeleton.npy')]
    results = []
    
    # We use ThreadPoolExecutor here to avoid the BrokenProcessPool error on Windows/Jupyter
    # It allows workers to access functions defined in the notebook cells directly
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = [executor.submit(process_single_file_metadata, f) for f in file_list]
        
        for future in tqdm(as_completed(futures), total=len(futures), desc="Building Registry (Parallel)"):
            try:
                res = future.result()
                if isinstance(res, dict):
                    results.append(res)
                elif isinstance(res, str):
                    print(f"\n[WARNING] {res}")
            except Exception as e:
                print(f"\n[ERROR] Thread failed: {e}")

    df = pd.DataFrame(results)
    df.to_csv(REGISTRY_PATH, index=False)
    return df

def get_variance(skeleton):
    """Calculate spatial variance to identify the dominant actor."""
    if skeleton is None or np.all(skeleton == 0): 
        return 0
    return np.var(skeleton, axis=(0, 1)).sum()

def fast_interpolate(skeleton):
    """NumPy-based linear interpolation. Significant speedup over Pandas."""
    f, j, c = skeleton.shape
    new_skel = skeleton.copy()
    for j_idx in range(j):
        for c_idx in range(c):
            coords = skeleton[:, j_idx, c_idx]
            invalid = (coords == 0) | np.isnan(coords)
            if not np.any(invalid) or np.all(invalid): 
                continue
            
            valid_idx = np.where(~invalid)[0]
            invalid_idx = np.where(invalid)[0]
            new_skel[invalid, j_idx, c_idx] = np.interp(invalid_idx, valid_idx, coords[valid_idx])
    return new_skel

def clean_and_normalize(skeleton, is_secondary=False, origin_traj=None):
    """Interpolates, smooths, and centers a skeleton."""
    # 1. Cleaning
    skeleton = fast_interpolate(skeleton)
    skeleton = medfilt(skeleton, kernel_size=(3, 1, 1))
    
    # 2. Normalization (Spatial Centering)
    if not is_secondary:
        origin_traj = skeleton[:, 0:1, :] # Center on SpineBase (Joint 0)
    
    return skeleton - origin_traj, origin_traj

# --- Worker Logic ---

def process_file_worker(row):
    """
    Worker function for ThreadPool. 
    Handles both Solo and Interaction logic based on row['is_interaction'].
    """
    fname = row['file_name']
    fpath = os.path.join(NTU_RAW_DATA_DIR, fname)
    
    try:
        data = np.load(fpath, allow_pickle=True).item()
        body_keys = [k for k in data.keys() if 'skel_body' in k]
        
        # Identify bodies by variance (highest = main actor)
        variances = {k: get_variance(data[k]) for k in body_keys}
        sorted_keys = sorted(variances, key=variances.get, reverse=True)

        if not row['is_interaction']:
            # SOLO LOGIC
            if len(sorted_keys) == 0:
                return False, f"{fname}: No bodies found."
            skel, _ = clean_and_normalize(data[sorted_keys[0]])
            output = {"skel_body0": skel, "action": row['A']}
        else:
            # INTERACTIVE LOGIC
            if len(sorted_keys) < 2:
                return False, f"{fname}: Expected 2 bodies, found {len(sorted_keys)}."
            
            # Check 50% presence for second body
            skel2_raw = data[sorted_keys[1]]
            presence = np.sum(~np.all(skel2_raw == 0, axis=(1, 2))) / len(skel2_raw)
            if presence < 0.5:
                return False, f"{fname}: Body2 presence only {presence:.2%}"

            skel1, origin = clean_and_normalize(data[sorted_keys[0]])
            skel2, _ = clean_and_normalize(data[sorted_keys[1]], is_secondary=True, origin_traj=origin)
            output = {"skel_body0": skel1, "skel_body1": skel2, "action": row['A']}
            
        # Save to processed directory
        np.save(os.path.join(PROC_DIR, fname), output)
        return True, None

    except Exception as e:
        return False, f"{fname}: {str(e)}"

# --- Parallel Runners ---

def run_pipeline(df, num_workers, mode='solo'):
    """Parallelized runner using ThreadPoolExecutor for Jupyter/Windows compatibility."""
    print(f"\n>>> Starting {mode.upper()} Pipeline ({num_workers} threads)")
    
    results_log = []
    # ThreadPool is safe for Jupyter and avoids BrokenProcessPool
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(process_file_worker, row): row['file_name'] 
                   for _, row in df.iterrows()}
        
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"Processing {mode}"):
            success, error = future.result()
            if not success:
                results_log.append(error)
                # print(f"\n[SKIP] {error}") # Uncomment to see specific file errors
            else:
                results_log.append("SUCCESS")
                
    return results_log

# 2. Apply Preprocessing Filter Criteria (Before Transformation)
def get_clean_subset(df):
    return df[
        (df['n_frames'] >= 30) & 
        (df['n_frames'] <= 200) & 
        (df['too_fast_torso'] == False)
    ].copy()

# DEBUG REMOVE FILES
reset_preprocessing_env()

print(f"Building auxillary csv at {REGISTRY_PATH}")
df_master = build_master_registry_parallel(CORE_USAGE_ALLOWED)
print(f"Built auxillary csv of size {df_master.shape}")

# Split Dataframes
df_solo = df_master[df_master['is_interaction'] == False].copy()
df_interaction = df_master[df_master['is_interaction'] == True].copy()

df_solo_clean = get_clean_subset(df_solo)
df_interaction_clean = get_clean_subset(df_interaction)

print(f"Split into df_solo and df_interaction, then cleaned by frame length outliers")
print(f"df_solo.shape = {df_solo_clean.shape}")
print(f"df_interaction = {df_interaction_clean.shape}")

# --- Solo ---
print(f"Processing df_solo_clean")
solo_results = run_pipeline(df_solo_clean, CORE_USAGE_ALLOWED, mode='solo')
print(f"Solo Processed: {len(solo_results)}")

# --- Interaction ---
print(f"Processing df_interaction_clean")
interaction_results = run_pipeline(df_interaction_clean, CORE_USAGE_ALLOWED, mode='interaction')
print(f"Interaction Processed: {len(interaction_results)}")

# --- Results ---

# View the results
print("\nSolo Head:")
print(df_solo_clean.head())
print("\nInteraction Head:")
print(df_interaction_clean.head())

Building auxillary csv at ntu_preprocess_master_registry.csv
Loading existing registry...
Built auxillary csv of size (56880, 8)
Split into df_solo and df_interaction, then cleaned by frame length outliers
df_solo.shape = (37933, 8)
df_interaction = (8519, 8)
Processing df_solo_clean

>>> Starting SOLO Pipeline (24 threads)


Processing solo:  30%|███       | 11503/37933 [00:36<01:23, 316.52it/s]
